In [17]:
import pandas as pd
import numpy as np

df = pd.read_csv("SURYA_Final_Data.csv")

print(df.columns)

print(df.head(n=10))

Index(['Unnamed: 0.1', 'Unnamed: 0', 'window_begin', 'window_end',
       '>10.0 MeV 10.0 pfu SEP Start Time', 'OSEP_label',
       'Flare_log_Strength_max', 'Flare_label', 'flare_start_time',
       'flare_peak_time', 'flare_end_time', 'flare_fl_cls', 'flare_Strength'],
      dtype='object')
   Unnamed: 0.1  Unnamed: 0 window_begin  window_end  \
0             0           0   2015-03-20  2015-03-21   
1             1           1   2012-02-09  2012-02-10   
2             2           2   2023-03-18  2023-03-19   
3             3           3   2021-10-26  2021-10-27   
4             4           4   2015-06-21  2015-06-22   
5             5           5   2020-12-05  2020-12-06   
6             6           6   2017-01-12  2017-01-13   
7             7           7   2017-09-10  2017-09-11   
8             8           8   2018-06-21  2018-06-22   
9             9           9   2011-06-07  2011-06-08   

  >10.0 MeV 10.0 pfu SEP Start Time  OSEP_label  Flare_log_Strength_max  \
0             

In [18]:
import s3fs
fs = s3fs.S3FileSystem(anon=True)

# Browse the structure
print(fs.ls("nasa-surya-bench/"))

['nasa-surya-bench/2010', 'nasa-surya-bench/2011', 'nasa-surya-bench/2012', 'nasa-surya-bench/2013', 'nasa-surya-bench/2014', 'nasa-surya-bench/2015', 'nasa-surya-bench/2016', 'nasa-surya-bench/2017', 'nasa-surya-bench/2018', 'nasa-surya-bench/2019', 'nasa-surya-bench/2020', 'nasa-surya-bench/2021', 'nasa-surya-bench/2022', 'nasa-surya-bench/2023', 'nasa-surya-bench/2024', 'nasa-surya-bench/index.html']


In [19]:
df = df.rename(columns={"OSEP_label": "SEP"})

df = df.rename(columns={"Flare_log_Strength_max": "flare_strength"})

df = df.rename(columns={"flare_start_time": "timestep"})

In [20]:
df = df.drop(columns=['Unnamed: 0','Unnamed: 0.1'])

In [21]:
sep_dates = df['timestep'].tolist()
print(len(sep_dates))
s3_paths = []
match_types = []

for date_str in sep_dates:
    dt = pd.to_datetime(date_str)
    
    # Subtract 3 hours (first preference)
    dt_primary = dt - pd.Timedelta(hours=3)
    
    # Fallback window: (dt - 5hr) to (dt - 1hr)
    dt_window_start = dt - pd.Timedelta(hours=5)
    dt_window_end   = dt - pd.Timedelta(hours=1)

    # Collect unique prefixes the window might span
    candidate_hours = [dt - pd.Timedelta(hours=h) for h in range(1, 6)]
    unique_prefixes = set(
        f"nasa-surya-bench/{t.year}/{t.month:02d}/" for t in candidate_hours
    )

    matched_file = None
    match_type   = None

    try:
        # Gather all files across relevant prefixes
        all_files = []
        for prefix in unique_prefixes:
            try:
                all_files.extend(fs.ls(prefix))
            except Exception:
                pass

        # Debug: print a few filenames to verify format (remove after confirming)
        if all_files:
            print(f"Sample files for {dt}: {all_files[:3]}")

        # --- First preference: exact match at dt - 3hrs ---
        # Format: 20150320_0057  (%Y%m%d_%H%M, no seconds)
        primary_str = dt_primary.strftime('%Y%m%d_%H%M')
        primary_matches = [f for f in all_files if primary_str in f]
        
        if primary_matches:
            matched_file = primary_matches[0]
            match_type   = "exact_minus3h"

        # --- Second preference: closest file within (dt-5h) to (dt-1h) ---
        if not matched_file:
            window_matches = []
            for f in all_files:
                fname = f.split('/')[-1]  # e.g. "20230318_0000.nc"
                try:
                    # Parse using the known format: YYYYMMDD_HHMM
                    file_dt = pd.to_datetime(fname[:13], format='%Y%m%d_%H%M')
                    if dt_window_start <= file_dt <= dt_window_end:
                        window_matches.append((f, file_dt))
                except Exception:
                    continue

            if window_matches:
                # Pick closest to dt_primary
                window_matches.sort(key=lambda x: abs((x[1] - dt_primary).total_seconds()))
                matched_file = window_matches[0][0]
                match_type   = "window_minus5h_to_minus1h"

    except Exception as e:
        print(f"Error for {dt}: {e}")

    s3_paths.append(f"s3://{matched_file}" if matched_file else np.nan)
    match_types.append(match_type if matched_file else "no_match")

df['path']       = s3_paths
df['match_type'] = match_types

#print(df[['window_begin', 'path', 'match_type', 'OSEP_label', 'Flare_log_Strength_max']])
print("\nMatch type counts:")
print(df['match_type'].value_counts())

25


Sample files for 2015-03-20 00:57:00: ['nasa-surya-bench/2015/03/20150301_0000.nc', 'nasa-surya-bench/2015/03/20150301_0012.nc', 'nasa-surya-bench/2015/03/20150301_0024.nc']


Sample files for 2012-02-09 09:20:00: ['nasa-surya-bench/2012/02/20120201_0000.nc', 'nasa-surya-bench/2012/02/20120201_0012.nc', 'nasa-surya-bench/2012/02/20120201_0024.nc']


Sample files for 2023-03-18 07:10:00: ['nasa-surya-bench/2023/03/20230301_0000.nc', 'nasa-surya-bench/2023/03/20230301_0012.nc', 'nasa-surya-bench/2023/03/20230301_0024.nc']


Sample files for 2021-10-26 02:40:00: ['nasa-surya-bench/2021/10/20211001_0000.nc', 'nasa-surya-bench/2021/10/20211001_0012.nc', 'nasa-surya-bench/2021/10/20211001_0024.nc']


...

Sample files for 2012-07-23 11:21:00: ['nasa-surya-bench/2012/07/20120701_0000.nc', 'nasa-surya-bench/2012/07/20120701_0012.nc', 'nasa-surya-bench/2012/07/20120701_0024.nc']


Sample files for 2014-04-01 00:43:00: ['nasa-surya-bench/2014/03/20140301_0000.nc', 'nasa-surya-bench/2014/03/20140301_0012.nc', 'nasa-surya-bench/2014/03/20140301_0024.nc']


Sample files for 2010-05-07 07:29:00: ['nasa-surya-bench/2010/05/20100513_0012.nc', 'nasa-surya-bench/2010/05/20100513_0024.nc', 'nasa-surya-bench/2010/05/20100513_0036.nc']


Sample files for 2023-05-09 03:42:00: ['nasa-surya-bench/2023/05/20230501_0000.nc', 'nasa-surya-bench/2023/05/20230501_0012.nc', 'nasa-surya-bench/2023/05/20230501_0024.nc']



Match type counts:
match_type
window_minus5h_to_minus1h    21
no_match                      3
exact_minus3h                 1
Name: count, dtype: int64


In [22]:
df

,window_begin,window_end,>10.0 MeV 10.0 pfu SEP Start Time,SEP,flare_strength,Flare_label,timestep,flare_peak_time,flare_end_time,flare_fl_cls,flare_Strength,path,match_type
0,2015-03-20,2015-03-21,NaN,0,-4.102373,1,2015-03-20 00:57:00,2015-03-20 01:33:00,2015-03-20 02:11:00,C,0.000079,s3://nasa-surya-bench/2015/03/20150319_2200.nc,window_minus5h_to_minus1h
1,2012-02-09,2012-02-10,NaN,0,-4.958607,1,2012-02-09 09:20:00,2012-02-09 09:33:00,2012-02-09 09:50:00,C,0.000011,s3://nasa-surya-bench/2012/02/20120209_0624.nc,window_minus5h_to_minus1h
2,2023-03-18,2023-03-19,NaN,0,-4.026872,1,2023-03-18 07:10:00,2023-03-18 07:16:00,2023-03-18 07:20:00,C,0.000094,s3://nasa-surya-bench/2023/03/20230318_0412.nc,window_minus5h_to_minus1h
3,2021-10-26,2021-10-27,NaN,0,-3.886057,1,2021-10-26 02:40:00,2021-10-26 02:47:00,2021-10-26 02:54:00,M,0.000130,s3://nasa-surya-bench/2021/10/20211025_2336.nc,window_minus5h_to_minus1h
4,2015-06-21,2015-06-22,2015/6/21 20:35,1,-3.420216,1,2015-06-21 09:38:00,2015-06-21 09:44:00,2015-06-21 09:50:00,M,0.000380,s3://nasa-surya-bench/2015/06/20150621_0636.nc,window_minus5h_to_minus1h
5,2020-12-05,2020-12-06,NaN,0,-4.795880,1,2020-12-05 00:16:00,2020-12-05 00:38:00,2020-12-05 00:52:00,C,0.000016,s3://nasa-surya-bench/2020/12/20201204_2112.nc,window_minus5h_to_minus1h
6,2017-01-12,2017-01-13,NaN,0,-4.420216,1,2017-01-12 15:54:00,2017-01-12 16:18:00,2017-01-12 16:41:00,C,0.000038,s3://nasa-surya-bench/2017/01/20170112_1248.nc,window_minus5h_to_minus1h
7,2017-09-10,2017-09-11,2017/9/10 16:45,1,-2.086186,1,2017-09-10 15:35:00,2017-09-10 16:06:00,2017-09-10 16:31:00,X,0.008200,s3://nasa-surya-bench/2017/09/20170910_1236.nc,window_minus5h_to_minus1h
8,2018-06-21,2018-06-22,NaN,0,-4.677781,1,2018-06-21 01:09:00,2018-06-21 01:15:00,2018-06-21 01:18:00,C,0.000021,s3://nasa-surya-bench/2018/06/20180620_2212.nc,window_minus5h_to_minus1h
9,2011-06-07,2011-06-08,2011/6/7 8:20,1,-3.602060,1,2011-06-07 06:16:00,2011-06-07 06:41:00,2011-06-07 06:59:00,M,0.000250,s3://nasa-surya-bench/2011/06/20110607_0312.nc,window_minus5h_to_minus1h


In [16]:
# STEP 1: Check what your timestamps look like after conversion
dt = pd.to_datetime(df['timestep'].iloc[0])
dt_primary = dt - pd.Timedelta(hours=3)
print(f"Original timestamp : {dt}")
print(f"After -3hrs        : {dt_primary}")
print(f"Primary string     : {dt_primary.strftime('%Y%m%d_%H%M')}")

# STEP 2: Check what prefix we're looking at
prefix = f"nasa-surya-bench/{dt_primary.year}/{dt_primary.month:02d}/"
print(f"\nLooking in prefix  : {prefix}")

# STEP 3: List actual files in that prefix
files = fs.ls(prefix)
print(f"\nTotal files found  : {len(files)}")
print(f"\nFirst 10 files:")
for f in files[:10]:
    print(f"  {f}")

Original timestamp : 2015-03-20 00:57:00
After -3hrs        : 2015-03-19 21:57:00
Primary string     : 20150319_2157

Looking in prefix  : nasa-surya-bench/2015/03/


NameError: name 'fs' is not defined

In [24]:
df['present']=1

df.loc[df['path'].isna(), 'present'] = 0

df.to_csv("SURYA_data_v3.csv")